# Préparation des données

In [56]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [57]:
# Chargements des datasets 

hotel_booking = pd.read_csv("../data/raw/hotel_bookings.csv")
flight_clean = pd.read_csv("../data/raw/Clean_Dataset.csv")

## Préparation du dataset Hôtel

In [58]:
hotel_booking.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


### Suppression des colonnes inutiles

La variable `company` contient plus de 94 % de valeurs manquantes. Cette proportion est trop importante pour permettre une exploitation fiable de cette variable.

Elle est donc supprimée afin de simplifier le jeu de données.

In [59]:
hotel_booking.drop(columns=["company"], inplace=True)

La variable `reservation_status` correspond au statut final de la réservation. Cette information n'est connue qu'après l'issue du séjour et est directement liée à la variable cible `is_canceled`.

La conserver entraînerait une fuite de données (data leakage), le modèle ayant accès à une information indisponible au moment de la prédiction.

Elle est donc supprimée avant la phase de modélisation.

In [60]:
hotel_booking.drop(columns=["reservation_status"], inplace=True)

In [61]:
hotel_booking.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status_date'],
      dtype='str')

### Traitement des valeurs manquantes

Les variables `children`, `country` et `agent` présentent quelques valeurs manquantes.

Les choix suivants sont retenus :

- `children` : remplacement par la médiane, car il s'agit d'une variable numérique.
- `country` : remplacement par la valeur "Unknown".
- `agent` : remplacement par 0 afin d'indiquer l'absence d'agent de réservation.

In [62]:
hotel_booking["children"] = hotel_booking["children"].fillna(
    hotel_booking["children"].median()
)

hotel_booking["country"] = hotel_booking["country"].fillna(
    "Unknown"
)

hotel_booking["agent"] = hotel_booking["agent"].fillna(
    0
)

In [63]:
# Vérification
hotel_booking.isnull().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces 

### Suppression des doublons

L'analyse exploratoire a mis en évidence la présence de réservations dupliquées.

Ces doublons sont supprimés afin d'éviter qu'une même observation n'influence plusieurs fois l'entraînement des futurs modèles de machine learning.

Le traitement des valeurs manquantes est désormais terminé. Une vérification confirme qu'aucune valeur manquante ne subsiste dans le jeu de données.

In [64]:
# Vérification du nombre de doublons
hotel_booking.duplicated().sum()

np.int64(32003)

In [65]:
# Suppression des doublons 
hotel_booking = hotel_booking.drop_duplicates()

In [66]:
# Vrérification après suppression 
hotel_booking.duplicated().sum()

np.int64(0)

In [67]:
# Vérification que le nombre de lignes sur le jeu de données a bien diminué
hotel_booking.shape

(87387, 30)

Après suppression des doublons, le jeu de données contient désormais **87 389 observations** réparties sur **31 variables**.

Cette étape permet d'éviter que certaines réservations soient surreprésentées lors de l'entraînement des modèles de machine learning.

### Features Engineering

Afin d'améliorer les performances des futurs modèles de machine learning, de nouvelles variables sont créées à partir des informations déjà présentes dans le jeu de données.

Cette étape, appelée *Feature Engineering*, permet de construire des variables plus représentatives du comportement des clients et susceptibles d'apporter davantage d'information aux modèles prédictifs.

Deux nouvelles variables sont créées :

- **total_nights** : représente le nombre total de nuits réservées, obtenu en additionnant les nuits passées en semaine et le week-end.
- **total_guests** : représente le nombre total de voyageurs en additionnant le nombre d'adultes, d'enfants et de bébés.

Ces variables offrent une vision plus synthétique de la réservation et pourront constituer des facteurs explicatifs pertinents pour la prédiction des annulations.

In [68]:
# Création de la variable "nombre total de nuits"
hotel_booking["total_nights"] = (
    hotel_booking["stays_in_weekend_nights"]
    + hotel_booking["stays_in_week_nights"]
)

In [69]:
# Création de la variable "nombre total de voyageurs"
hotel_booking["total_guests"] = (
    hotel_booking["adults"]
    + hotel_booking["children"]
    + hotel_booking["babies"]
)

In [70]:
# Vérification du dataset après l'ajout de ces variables 
hotel_booking[["total_nights", "total_guests"]].head()

,total_nights,total_guests
0,0,2.0
1,0,2.0
2,1,1.0
3,1,1.0
4,2,2.0


Les variables **total_nights** et **total_guests** ont été créées avec succès.

Elles permettent de résumer des informations importantes sur la réservation en regroupant plusieurs variables existantes. Ces nouvelles caractéristiques pourront être utilisées lors de l'entraînement des modèles afin d'améliorer leur capacité de prédiction.

### Encodage des variables catégorielles

Les algorithmes de machine learning ne peuvent pas traiter directement les variables de type texte.

Les variables catégorielles sont donc transformées en valeurs numériques grâce à la méthode **Label Encoding**.

Cette transformation permet de conserver l'information tout en rendant le jeu de données exploitable par les futurs modèles.

In [71]:
hotel_booking.select_dtypes(include="object").columns

/var/folders/g6/_fb35h6n4r7gskyw19rk12900000gn/T/ipykernel_1963/1498786997.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  hotel_booking.select_dtypes(include="object").columns


Index(['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment',
       'distribution_channel', 'reserved_room_type', 'assigned_room_type',
       'deposit_type', 'customer_type', 'reservation_status_date'],
      dtype='str')

In [72]:
encoder = LabelEncoder()

colonnes = hotel_booking.select_dtypes(include="object").columns.drop("reservation_status_date")

for col in colonnes:
    hotel_booking[col] = encoder.fit_transform(hotel_booking[col])

/var/folders/g6/_fb35h6n4r7gskyw19rk12900000gn/T/ipykernel_1963/1044843487.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes = hotel_booking.select_dtypes(include="object").columns.drop("reservation_status_date")


In [73]:
hotel_booking.info()

<class 'pandas.DataFrame'>
Index: 87387 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87387 non-null  int64  
 1   is_canceled                     87387 non-null  int64  
 2   lead_time                       87387 non-null  int64  
 3   arrival_date_year               87387 non-null  int64  
 4   arrival_date_month              87387 non-null  int64  
 5   arrival_date_week_number        87387 non-null  int64  
 6   arrival_date_day_of_month       87387 non-null  int64  
 7   stays_in_weekend_nights         87387 non-null  int64  
 8   stays_in_week_nights            87387 non-null  int64  
 9   adults                          87387 non-null  int64  
 10  children                        87387 non-null  float64
 11  babies                          87387 non-null  int64  
 12  meal                            87387 non-null 

In [74]:
hotel_booking.select_dtypes(include="object").columns

/var/folders/g6/_fb35h6n4r7gskyw19rk12900000gn/T/ipykernel_1963/1498786997.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  hotel_booking.select_dtypes(include="object").columns


Index(['reservation_status_date'], dtype='str')

La variable `reservation_status_date` correspond à la date à laquelle une réservation a été annulée, confirmée ou terminée.

Cette information n'est connue qu'après l'issue de la réservation. Son utilisation entraînerait une fuite de données (*data leakage*) et biaiserait les performances des modèles de machine learning.

Elle est donc supprimée avant la phase de modélisation.

In [75]:
hotel_booking.drop(columns="reservation_status_date", inplace=True)

In [76]:
hotel_booking.info()

<class 'pandas.DataFrame'>
Index: 87387 entries, 0 to 119389
Data columns (total 31 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   hotel                           87387 non-null  int64  
 1   is_canceled                     87387 non-null  int64  
 2   lead_time                       87387 non-null  int64  
 3   arrival_date_year               87387 non-null  int64  
 4   arrival_date_month              87387 non-null  int64  
 5   arrival_date_week_number        87387 non-null  int64  
 6   arrival_date_day_of_month       87387 non-null  int64  
 7   stays_in_weekend_nights         87387 non-null  int64  
 8   stays_in_week_nights            87387 non-null  int64  
 9   adults                          87387 non-null  int64  
 10  children                        87387 non-null  float64
 11  babies                          87387 non-null  int64  
 12  meal                            87387 non-null 

In [77]:
# Sauvegarde pour modélisation 
hotel_booking.to_csv("../data/processed/hotel_booking_prepared.csv", index=False)

## Préparation du dataset vols

Comme pour les hôtels, cette partie du notebook a pour objectif de préparer le jeu de données des vols avant la phase de modélisation.

Les principales étapes sont :
- suppression des colonnes inutiles ;
- vérification des valeurs manquantes ;
- suppression des doublons ;
- encodage des variables catégorielles ;
- sauvegarde du jeu de données préparé.

In [78]:
flight_clean.head()

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


### Suppression des colonnes inutiles
La colonne `Unnamed: 0` correspond à un ancien index généré lors de l'export du jeu de données.

Elle n'apporte aucune information utile et est supprimée.

In [79]:
flight_clean.drop(columns="Unnamed: 0", inplace=True)

In [80]:
flight_clean.head()

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955


### Valeurs manquantes
Une vérification est effectuée afin de s'assurer qu'aucune valeur manquante n'est présente dans le jeu de données.

Le contrôle confirme que toutes les variables sont complètes. Aucun traitement supplémentaire n'est donc nécessaire.

In [81]:
flight_clean.isnull().sum()

airline             0
flight              0
source_city         0
departure_time      0
stops               0
arrival_time        0
destination_city    0
class               0
duration            0
days_left           0
price               0
dtype: int64

### Doublons

Une vérification est réalisée afin de détecter d'éventuels doublons dans le jeu de données.

Le contrôle confirme qu'aucune observation dupliquée n'est présente. Aucune suppression n'est donc nécessaire.

In [82]:
flight_clean.duplicated().sum()

np.int64(0)

### Encodage des variables catégorielles

Les modèles de machine learning ne peuvent pas utiliser directement des variables textuelles.

Les variables catégorielles sont donc transformées en valeurs numériques à l'aide du **Label Encoding**.

Les colonnes concernées sont :
- `airline`
- `flight`
- `source_city`
- `departure_time`
- `stops`
- `arrival_time`
- `destination_city`
- `class`

In [83]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

colonnes_categorielles = flight_clean.select_dtypes(include="object").columns

for col in colonnes_categorielles:
    flight_clean[col] = encoder.fit_transform(flight_clean[col])

/var/folders/g6/_fb35h6n4r7gskyw19rk12900000gn/T/ipykernel_1963/1108985381.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_categorielles = flight_clean.select_dtypes(include="object").columns


In [84]:
flight_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 300153 entries, 0 to 300152
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   airline           300153 non-null  int64  
 1   flight            300153 non-null  int64  
 2   source_city       300153 non-null  int64  
 3   departure_time    300153 non-null  int64  
 4   stops             300153 non-null  int64  
 5   arrival_time      300153 non-null  int64  
 6   destination_city  300153 non-null  int64  
 7   class             300153 non-null  int64  
 8   duration          300153 non-null  float64
 9   days_left         300153 non-null  int64  
 10  price             300153 non-null  int64  
dtypes: float64(1), int64(10)
memory usage: 25.2 MB


In [85]:
# On vérifie que les variables catégorielles sont bien remplacées par des variables numériques
flight_clean.head()

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,4,1408,2,2,2,5,5,1,2.17,1,5953
1,4,1387,2,1,2,4,5,1,2.33,1,5953
2,0,1213,2,1,2,1,5,1,2.17,1,5956
3,5,1559,2,4,2,0,5,1,2.25,1,5955
4,5,1549,2,4,2,4,5,1,2.33,1,5955


In [86]:
# Sauvegarde pour modélisation
flight_clean.to_csv("../data/processed/flight_prepared.csv", index=False)